# H(p,q) workflow
This notebook demonstrates the analysis workflow using the pre-defined functions in `src/compute_H.py`.
It does not redefine any of the library functions -- it imports and reuses them -- and displays all generated images and saved grids from the `outputs/` directory.

In [ ]:
# Import existing functions and helpers
from gw_turbulence import *
import numpy as np
import os
import matplotlib.pyplot as plt
from IPython.display import display, Image
# Ensure inline plotting works when running interactively
%matplotlib inline

## Fast: generate analytic p$\\to$0 spectra 

In [ ]:
# Generate analytic spectra for a set of Mach numbers and save to outputs/
M_values = [0.001, 0.01, 0.1, 1.0]
out_dir = os.path.abspath('outputs')
os.makedirs(out_dir, exist_ok=True)
out_png = os.path.join(out_dir, 'H_spectra_analytic.png')

# Call the function - it will append M and R values (2-digit scientific notation) to the filename
plot_spectra_M_analytic(M_values, qmin=1e-4, qmax=1e1, nq=300, out_png=out_png, R=1e4)

# Construct the actual filename with M and R appended (matching 2-digit scientific notation format)
mlist_str = '-'.join([f"{M:.2e}".replace('+', 'p').replace('-', 'm') for M in M_values])
rstr = f"{1e4:.2e}".replace('+', 'p').replace('-', 'm')
out_png_base, out_png_ext = os.path.splitext(out_png)
actual_png = f"{out_png_base}_Ms{mlist_str}_R{rstr}{out_png_ext}"

if os.path.exists(actual_png):
    display(Image(filename=actual_png))
else:
    print(f"File not found: {actual_png}")


## Optional Decaying Grid With Live Progress
Use this cell to evaluate the decaying model directly in the notebook with live point-by-point progress output.

In [ ]:
use_mpi = mpi_is_active()
processes = 1 if use_mpi else 8
status = LiveStatusLogger(prefix="decaying-grid", every_seconds=0.0, root_only=True)
ps = np.logspace(-2, -0.5, 6)
qs = np.logspace(-2, -0.5, 6)

print(f"Running decaying grid with live progress... MPI active: {use_mpi}, processes: {processes}")
H_decaying = H_pq_decaying_grid(
    ps,
    qs,
    M=0.1,
    R=1e2,
    k0=1.0,
    verbose=True,
    convolution_method="trapz",
    convolution_points=48,
    integration_method="sampled",
    x_points=16,
    y_points=16,
    status=status,
    log_points=True,
    use_mpi=use_mpi,
    processes=processes,
)

# MPI usage from a terminal:
# mpiexec -n 4 python -m src.gw_turbulence.cli --use-mpi --scan-points 6
# or inside Python started under mpiexec set use_mpi=True below.

plt.figure(figsize=(6, 4))
plt.pcolormesh(ps, qs, H_decaying, shading='auto', cmap='viridis')
plt.xscale('log')
plt.yscale('log')
plt.xlabel('p = k/k0')
plt.ylabel('q = ω/k0')
plt.colorbar(label='H decaying')
plt.tight_layout()
plt.show()


## Display all PNG images in `outputs/`
This will show all existing PNGs found in the repository `outputs/` directory.

In [ ]:
# Display every PNG in outputs/
out_dir = os.path.abspath('outputs')
pngs = sorted([f for f in os.listdir(out_dir) if f.lower().endswith('.png')])
print(f'Found {len(pngs)} PNG files in', out_dir)
for fn in pngs:
    path = os.path.join(out_dir, fn)
    print(' ---', fn, '---')
    display(Image(filename=path))

## Visualize saved H grids (.npz)
If the `.npz` files were saved with keys `ps`, `qs`, and `H`, this cell will load and plot them.

In [ ]:
# Load and visualize any Hgrid .npz files in outputs/
npz_files = sorted([f for f in os.listdir(out_dir) if f.lower().endswith('.npz')])
for fn in npz_files:
    path = os.path.join(out_dir, fn)
    print(' ---', fn, '---')
    try:
        data = np.load(path)
        if not {'ps', 'qs', 'H'}.issubset(data.files):
            print('Skipping', fn, '— missing expected keys, found:', list(data.files))
            continue
        ps = data['ps']
        qs = data['qs']
        H = data['H']
        # Plot H grid
        plt.figure(figsize=(6, 4))
        plt.pcolormesh(ps, qs, H, shading='auto', cmap='viridis')
        plt.xscale('log')
        plt.yscale('log')
        plt.xlabel('p = k/k0')
        plt.ylabel(r'q = $\omega$/k0')
        plt.title(fn)
        plt.colorbar(label='H')
        plt.tight_layout()
        display(plt.gcf())
        plt.close()
    except Exception as e:
        print('Error loading', fn, e)

---
### Notes and next steps
- The analytic spectra generation uses `plot_spectra_M_analytic` (fast).
- The full 2D scan `example_scan_and_plot` is provided but left commented because it can be very slow.
- If you want, I can run the optional 2D scan cell, or adjust plotting styles / export formats.